# Topic modeling + Doc2Vec on LinkedIn job postings

## Overview

In this notebook, I apply unsupervised learning techniques to discover hidden patterns in a [LinkedIn job postings dataset from KaggleHub.

To do this I:
1. Preprocess a corpus using spaCy tokenization, lemmatization, and stop word removal.
2. Build a Gensim Dictionary and convert documents into Bag-of-Words and TF-IDF representations.
3. Apply Latent Dirichlet Allocation (LDA) to discover topic clusters.
4. Build a matching system using both TF-IDF cosine similarity and Doc2Vec embeddings.
5. Explore semantic relationships using pre-trained GloVe word embeddings.

#### Dataset

[LinkedIn Job Postings dataset](https://www.kaggle.com/datasets/arshkon/linkedin-job-postings) contains job descriptions and other data from various postings.

*Arsh Koneru. (2024). LinkedIn Job Postings (2023 - 2024) [Data set]. Kaggle. https://doi.org/10.34740/KAGGLE/DSV/9200871*


In [ ]:
!pip install gensim -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 59.1 MB/s eta 0:00:00


In [1]:
import random
import textwrap

import numpy as np
import pandas as pd
import spacy

import kagglehub

import gensim
import gensim.downloader as api
from gensim import corpora
from gensim import models
from gensim import similarities
from gensim.models import doc2vec

ModuleNotFoundError: No module named 'gensim'

In [ ]:
# set random seed
seed = 42
# Load spacy small english language model
nlp = spacy.load("en_core_web_sm")

### Download and Convert to DataFrame

Load the dataset and, if necessary, convert it to a Pandas DataFrame.

In [ ]:
directory = kagglehub.dataset_download("arshkon/linkedin-job-postings")
directory

100%|██████████| 159M/159M [00:01<00:00, 115MB/s]

Extracting files...


'/root/.cache/kagglehub/datasets/arshkon/linkedin-job-postings/versions/13'

In [ ]:
# Read dataset into pandas
file_path = f"{directory}/postings.csv"
df = pd.read_csv(file_path)
df.shape

(123849, 31)

In [ ]:
# Inspect the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   object 
 6   location                    123849 non-null  object 
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  object 
 12  applies                     23320 non-null   float64
 13  original_liste

In [ ]:
df.describe()

,job_id,max_salary,company_id,views,med_salary,min_salary,applies,original_listed_time,remote_allowed,expiry,closed_time,listed_time,sponsored,normalized_salary,zip_code,fips
count,1.238490e+05,2.979300e+04,1.221320e+05,122160.000000,6280.000000,2.979300e+04,23320.000000,1.238490e+05,15246.0,1.238490e+05,1.073000e+03,1.238490e+05,123849.0,3.607300e+04,102977.000000,96434.000000
mean,3.896402e+09,9.193942e+04,1.220401e+07,14.618247,22015.619876,6.491085e+04,10.591981,1.713152e+12,1.0,1.716213e+12,1.712928e+12,1.713204e+12,0.0,2.053270e+05,50400.491887,28713.879887
std,8.404355e+07,7.011101e+05,2.554143e+07,85.903598,52255.873846,4.959738e+05,29.047395,4.848209e+08,0.0,2.321394e+09,3.622893e+08,3.989122e+08,0.0,5.097627e+06,30252.232515,16015.929825
min,9.217160e+05,1.000000e+00,1.009000e+03,1.000000,0.000000,1.000000e+00,1.000000,1.701811e+12,1.0,1.712903e+12,1.712346e+12,1.711317e+12,0.0,0.000000e+00,1001.000000,1003.000000
25%,3.894587e+09,4.828000e+01,1.435200e+04,3.000000,18.940000,3.700000e+01,1.000000,1.712863e+12,1.0,1.715481e+12,1.712670e+12,1.712886e+12,0.0,5.200000e+04,24112.000000,13121.000000
50%,3.901998e+09,8.000000e+04,2.269650e+05,4.000000,25.500000,6.000000e+04,3.000000,1.713395e+12,1.0,1.716042e+12,1.712670e+12,1.713408e+12,0.0,8.150000e+04,48059.000000,29183.000000
75%,3.904707e+09,1.400000e+05,8.047188e+06,8.000000,2510.500000,1.000000e+05,8.000000,1.713478e+12,1.0,1.716088e+12,1.713283e+12,1.713484e+12,0.0,1.250000e+05,78201.000000,42077.000000
max,3.906267e+09,1.200000e+08,1.034730e+08,9975.000000,750000.000000,8.500000e+07,967.000000,1.713573e+12,1.0,1.729125e+12,1.713562e+12,1.713573e+12,0.0,5.356000e+08,99901.000000,56045.000000


In [ ]:
df[15:20]

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
15,83789755,OsteoStrong,Osteogenic Loading Coach,"Company DescriptionOsteoStrong® is not a gym, ...",NaN,NaN,"Anchorage, AK",3810432.0,4.0,NaN,...,NaN,1.713466e+12,NaN,0,FULL_TIME,NaN,NaN,NaN,99501.0,2020.0
16,95428182,CLEVELAND KIDS BOOK BANK,Administrative Coordinator,Job Title: Administrative CoordinatorOrganizat...,NaN,HOURLY,"Cleveland, OH",55624331.0,1.0,25.0,...,NaN,1.712856e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,52000.0,44102.0,39035.0
17,103860943,NaN,Customer Service / Reservationist,Sentinel Limousine of East Providence RI is a ...,19.0,HOURLY,"Providence, RI",NaN,3.0,NaN,...,NaN,1.713532e+12,NaN,0,PART_TIME,USD,BASE_SALARY,38480.0,2903.0,44007.0
18,111513530,United Methodists of Greater New Jersey,"Content Writer, Communications","Application opening date: April 24, 2024\nTitl...",NaN,NaN,Greater Philadelphia,4028816.0,10.0,NaN,...,NaN,1.713457e+12,NaN,0,FULL_TIME,NaN,NaN,NaN,NaN,NaN
19,115639136,Shannon Waltchack,Controller,WORK @ SWShannon Waltchack (SW) is seeking a C...,NaN,NaN,"Birmingham, AL",988555.0,61.0,NaN,...,Strong interpersonal communication skills and ...,1.713210e+12,NaN,0,FULL_TIME,NaN,NaN,NaN,35203.0,1073.0


### Sample the Dataset

I sample 3000 rows from the dataset

In [ ]:
data_sample = df.sample(n=3000,random_state=42).reset_index(drop=True)

## Data Preparation and Preprocessing

In [ ]:
# Extract primary text column "description" from Job Postings dataset
#.. stored as a list of documents using .tolist()
job_descriptions = data_sample["description"].tolist()

### Preprocessing with spaCy

Tokenizer function is defined that processes text using spaCy. The function does the following:
- Removes punctuation
- Removes stop words
- Removes non-alphabetic tokens (numbers, special characters)
- Removes named entities (companies)
- Returns the lemmatized form of the remaining tokens

Note: named entities are removed so that companies that post multiple (non-identical) jobs are not mistakenly grouped together, given that we want to group by actual job roles (i.e. skills, responsibilities, etc.)

In [ ]:
# Tokenizer
def spacy_tokenizer(text: str) -> list[str]:
    doc = nlp(text.lower())
    return  [token.lemma_ for token in doc if token.is_alpha and token.ent_iob_ == "O"
        and not token.is_stop and len(token.lemma_) > 2]

In [ ]:
# Apply Tokenizer to all docs (job_descriptions)
# just apply documents to it in list comprehension (not a specific module)

processed_docs = [spacy_tokenizer(job) for job in job_descriptions]
processed_docs

[['senior',
  'automation',
  'power',
  'system',
  'engineer',
  'primarily',
  'responsible',
  'conception',
  'design',
  'development',
  'implementation',
  'electrical',
  'system',
  'candidate',
  'extensive',
  'knowledge',
  'electrical',
  'power',
  'system',
  'active',
  'end',
  'afe',
  'system',
  'plc',
  'programming',
  'experience',
  'candidate',
  'possess',
  'ability',
  'apply',
  'mathematical',
  'engineering',
  'principle',
  'detailed',
  'description',
  'responsible',
  'design',
  'develop',
  'electrical',
  'system',
  'include',
  'generator',
  'control',
  'switchboards',
  'drive',
  'active',
  'end',
  'afe',
  'motor',
  'control',
  'system',
  'panel',
  'board',
  'component',
  'require',
  'function',
  'system',
  'accord',
  'customer',
  'power',
  'system',
  'simulation',
  'analysis',
  'engineering',
  'complex',
  'power',
  'system',
  'microgrid',
  'large',
  'analyze',
  'harmonic',
  'corrective',
  'measure',
  'complex',


---

## Vectorization with TF-IDF

### Creating a Gensim Dictionary

Here a Gensim `Dictionary` is built from the processed documents. The dictionary maps every unique token to a unique integer ID, which allows for efficient representation of Gensim corpora.

In [ ]:
# can access like normal python dictionary (except keys are id)
docs_dict = corpora.Dictionary(processed_docs)

In [ ]:
# Job Postings dataset
for i in range(20):
  print(f"id: {i}: {docs_dict[i]}")

id: 0: ability
id: 1: accord
id: 2: active
id: 3: actual
id: 4: additional
id: 5: afe
id: 6: alert
id: 7: analysis
id: 8: analyze
id: 9: apply
id: 10: approach
id: 11: area
id: 12: autocad
id: 13: automation
id: 14: awareness
id: 15: base
id: 16: bend
id: 17: board
id: 18: bus
id: 19: business


### Filtering Extremes

Words that appear in fewer than 2 documents or more than %50 of the documents are removed.

In [ ]:
# gensim: dictName.filter_extremes(no_below=,no_above=)
docs_dict.filter_extremes(no_below=2, no_above=0.5)

### Creating Bag-of-Words Corpus

Processed documents are converted to a Bag-of-Words representation using the filtered dictionary.

In [ ]:
bow_corpus = [docs_dict.doc2bow(doc) for doc in processed_docs]
len(bow_corpus)

3000

### Generating TF-IDF Vectors

TF-IDF model is created from the Bag-of-Words corpus.

In [ ]:
# models.TfidfModel(corpusHere)
tfidf_model = models.TfidfModel(bow_corpus)

In [ ]:
# model is applied to the corpus to create TF-IDF
tfidf_corpus = tfidf_model[bow_corpus] #tfidf_model[] is triggering Gensim's __getitem__ method

---

## Topic Modeling with LDA

### Training the LDA Model

An LDA (Latent Dirichlet Allocation) model is trained on the TF-IDF corpus to discover latent clusters.


In [ ]:
# originally tried num_topics=10, passes=20 this didn't produce coherent results
# next tried 20 topics & 30 passes
# next tried 30 topics & 30 passes
lda_model = models.LdaModel(corpus=tfidf_corpus, id2word=docs_dict, num_topics=30, passes=30, random_state=42)

### Extracting Topic Keywords

Below the top 10 keywords for each discovered topic are displayed.

In [ ]:
lda_model.print_topics(num_topics=30, num_words=10)

[(0,
  '0.015*"skf" + 0.007*"specimen" + 0.006*"fedex" + 0.005*"autism" + 0.005*"nordstrom" + 0.004*"napa" + 0.004*"ace" + 0.004*"mom" + 0.003*"aba" + 0.003*"seattle"'),
 (1,
  '0.011*"galt" + 0.010*"credit" + 0.007*"sale" + 0.007*"figure" + 0.007*"earning" + 0.007*"saas" + 0.007*"owner" + 0.006*"personal" + 0.006*"membership" + 0.005*"careerstaff"'),
 (2,
  '0.012*"trading" + 0.005*"trader" + 0.004*"ios" + 0.003*"thrivework" + 0.002*"eci" + 0.002*"pcu" + 0.002*"altame" + 0.002*"massage" + 0.002*"quantitative" + 0.002*"hydro"'),
 (3,
  '0.011*"beauty" + 0.007*"trailer" + 0.006*"ulta" + 0.006*"bakery" + 0.005*"req" + 0.005*"tradesman" + 0.003*"hair" + 0.003*"street" + 0.003*"foodservice" + 0.003*"datadog"'),
 (4,
  '0.006*"leasing" + 0.004*"associatesassist" + 0.004*"bayada" + 0.002*"delta" + 0.002*"pierce" + 0.002*"tpm" + 0.002*"gate" + 0.002*"csat" + 0.002*"westside" + 0.002*"mcc"'),
 (5,
  '0.007*"hardware" + 0.006*"coordinator" + 0.006*"chain" + 0.006*"logistic" + 0.005*"troubleshoo

---

## Document Retrieval

In this section, I implement a query-based document retrieval system using two different approaches:
1. TF-IDF with cosine similarity (exact word overlap, weighted by rarity)
2. Doc2Vec embeddings (semantic meaning, capturing conceptual similarity)


The query created consists of 10 specific terms that represent a target subset of the documents in the dataset. I'm generally testing for jobs pertaining to: *Language Engineer, Data Linguist, Data Language Scientist, NLP Linguist, Machine Learning Language Scientist*, etc.

In [ ]:
query = [
    "python",
    "data",
    "engineer",
    "language",
    "learning",
    "stack",
    "database",
    "algorythm",
    "program",
    "machine"
]


# Vectorize query same as done for query the original documents
query_bow = docs_dict.doc2bow(query)

In [ ]:
# TF-IDF
# Use the `MatrixSimilarity` class to calculate cosine similarity between vectors.
query_tfidf = tfidf_model[query_bow]

In [ ]:
# Calculate the similarities of all documents to query.
index_tfidf = similarities.MatrixSimilarity(tfidf_corpus)
sims_tfidf = index_tfidf[query_tfidf] # results in np array of scores

In [ ]:
# Display the top 3 most similar documents based on TF-IDF similarity.
sorted_sims_tfidf = sorted(enumerate(sims_tfidf), key=lambda sim: -sim[1])

for rank in range(3):
  doc_index = sorted_sims_tfidf[rank][0]
  sim_score = sorted_sims_tfidf[rank][1]
  print(f"\n\nRank {rank + 1} (Score: {sim_score:.4f}):")
  print("\n".join(textwrap.wrap(job_descriptions[doc_index])))




Rank 1 (Score: 0.4036):
One of our Fortune 100 clients in Plano, TX is looking for a Full
Stack Data Engineer/Developer.   Title: Full Stack Data Engineer /
DeveloperLocation: Plano, TX (Fri – Remote)Hourly: $70 per hour
(W2)Duration: 1 year (Contract-to-Hire)  Stack Experience:Minimum 5
years of Full Stack expertise in one of the following stacks and
comfortable exploring othersMERN stack: JavaScript - MongoDB - Express
- ReactJS - Node.js (Preferred)MEAN stack: JavaScript - MongoDB -
Express - AngularJS - Node.jsLAMP stack: JavaScript - Linux - Apache -
MySQL – PHPLEMP stack: JavaScript - Linux - Nginx - MySQL – PHPDjango
stack: JavaScript - Python - Django – MySQLRuby on Rails: JavaScript -
Ruby - SQLite – Rails   Must-to-Have:Git URL(Required)Minimum 5 years
of experience with Data Modeling in Big Data environment and have
worked on massive structured/unstructured datasets beforeBig Data
stack (Hadoop, Hive, Spark, Kafka, Airflow/OOZIE,
BigQuery/Presto/Impala etc.)Experience work

### Matching via Doc2Vec

Next, query matches are found using **Doc2Vec** embeddings which captures semantic meaning, allowing it to find documents that are conceptually similar even if they use different vocabulary.

In [ ]:
tagged_docs = [doc2vec.TaggedDocument(words=doc, tags=[str(i)]) for i, doc in enumerate(processed_docs)]
len(tagged_docs)

3000

Doc2Vec model instantiated with the following parameters:
- `vector_size`: Dimensionality of the feature vectors (embeddings). Larger vectors can capture more nuance but require more training data.
- `min_count`: Ignores all words with total frequency lower than this value.
- `epochs`: Number of iterations over the corpus to use for training.

In [ ]:
d2v_model = doc2vec.Doc2Vec(vector_size=50, min_count=2, epochs=40)

In [ ]:
# build vocabulary for Doc2Vec
d2v_model.build_vocab(tagged_docs)

In [ ]:
# Display the first 20 words in the vocabulary by their index.
for i in range(20):
    print(f"ID: {i:2} | Word: {d2v_model.wv.index_to_key[i]}")

ID:  0 | Word: work
ID:  1 | Word: experience
ID:  2 | Word: team
ID:  3 | Word: include
ID:  4 | Word: customer
ID:  5 | Word: service
ID:  6 | Word: require
ID:  7 | Word: job
ID:  8 | Word: provide
ID:  9 | Word: skill
ID: 10 | Word: opportunity
ID: 11 | Word: business
ID: 12 | Word: company
ID: 13 | Word: support
ID: 14 | Word: time
ID: 15 | Word: ability
ID: 16 | Word: employee
ID: 17 | Word: management
ID: 18 | Word: position
ID: 19 | Word: need


In [ ]:
# Note to self: Check what are epochs..
d2v_model.train(tagged_docs, total_examples=d2v_model.corpus_count, epochs=d2v_model.epochs)

In [ ]:
# vector representation calculated for query
query_vec = d2v_model.infer_vector(query)
#print(query_vec)

Next, using the trained model the top 3 most similar documents are displayed

In [ ]:
sims_d2v = d2v_model.dv.most_similar([query_vec], topn=3)
# Doc2Vec (sims_d2v) returns a list of (string, float) tuples:
print(sims_d2v)

for rank, (tag, sim_score) in enumerate(sims_d2v[:3]):
  doc_index = int(tag)
  print(f"\n\nRank {rank + 1} (Score: {sim_score:.4f})")
  print(f"\n".join(textwrap.wrap(job_descriptions[doc_index])))



[('2933', 0.6148601174354553), ('1593', 0.5975397825241089), ('858', 0.5906465649604797)]


Rank 1 (Score: 0.6149)
One of our Fortune 100 clients in Plano, TX is looking for a Full
Stack Data Engineer/Developer.   Title: Full Stack Data Engineer /
DeveloperLocation: Plano, TX (Fri – Remote)Hourly: $70 per hour
(W2)Duration: 1 year (Contract-to-Hire)  Stack Experience:Minimum 5
years of Full Stack expertise in one of the following stacks and
comfortable exploring othersMERN stack: JavaScript - MongoDB - Express
- ReactJS - Node.js (Preferred)MEAN stack: JavaScript - MongoDB -
Express - AngularJS - Node.jsLAMP stack: JavaScript - Linux - Apache -
MySQL – PHPLEMP stack: JavaScript - Linux - Nginx - MySQL – PHPDjango
stack: JavaScript - Python - Django – MySQLRuby on Rails: JavaScript -
Ruby - SQLite – Rails   Must-to-Have:Git URL(Required)Minimum 5 years
of experience with Data Modeling in Big Data environment and have
worked on massive structured/unstructured datasets beforeBig Data
stac

---

## Exploring Semantics with GloVe

In this section, I explore some of the  relational operations with pre-trained GloVe (Global Vectors for Word Representation) embeddings.

In [ ]:
# Load Pre-Trained GloVe Embeddings
glove_vectors = api.load("glove-wiki-gigaword-50")

[==================================================] 100.0% 66.0/66.0MB downloaded


### Vector Arithmetic

Here **two different combinations** of terms using the `most_similar` method with `positive` and `negative` arguments

In [ ]:
result = glove_vectors.most_similar(positive=["engineer", "design"], negative=["software"], topn=5)
for word, similarity_score in result:
    print(f"{similarity_score:.3f} | {word}")

0.743 | architect
0.673 | commissioned
0.630 | sculptor
0.616 | conductor
0.614 | distinguished


In [ ]:
result2 = glove_vectors.most_similar(positive=["data", "technical"], negative=["engineer"], topn=5)
for word, similarity_score in result2:
    print(f"{similarity_score:.3f} | {word}")

0.732 | specific
0.721 | scope
0.710 | analysis
0.704 | assessing
0.700 | evaluating


### Odd One Out

Testing the effectiveness of `glove_vectors.doesnt_match()`

In [ ]:
words1 = ["python", "software", "engineer", "data", "snake"]
glove_vectors.doesnt_match(words1)

'snake'

In [ ]:
words2 = ["pipeline", "design", "workflow", "database", "education"]
glove_vectors.doesnt_match(words2)

'pipeline'

---

## Reflective Analysis Questions


### Question 1

When inspecting the LDA topics, which examples made you feel confident that the model found meaningful clusters, and which examples made you doubt the coherence of certain topics? How might you adjust the modeling process (such as number of topics or preprocessing) to improve coherence?


## Answer (Q1):
After testing with 20 and 30 passes, there was the only result that showed any sort of coherence. Every other one is seemingly a random assortments of different words. Interestingly, in each case the most coherent topics had to do with nursing (see below).

* _(Topic 3 with 10 topics)_
_after testing with 10 and 10 passes._
```
 (3,
  '0.026*"patient" + 0.017*"care" + 0.014*"nursing" + 0.012*"nurse" + 0.006*"hospital" + 0.006*"clinical" + 0.006*"physician" + 0.005*"therapist" + 0.005*"health" + 0.004*"therapy"')
```

* _(Topic 17 with 20 topics)_
_after testing with 20 and 30 passes_
```
 (17,
  '0.010*"nursing" + 0.009*"hospice" + 0.007*"nurse" + 0.005*"liberty" + 0.005*"psychiatric" + 0.004*"caregiver" + 0.004*"compassion" + 0.004*"caring" + 0.003*"counselor" + 0.003*"rehab"')
```

In addition to changing the number of passes, to try and improve it after my first run (which was even worse), I modified the tokenizer to filter out words that were less than 2 characters long (as there were tokens such as "h", "y", "de", etc.). Additionally, I added a filter to remove named entities (`token.ent_iob_`) as there were company names dispersed throughout the topic clusters that did not have any visible logic (i.e. the companies were not co-occurring with vocabulary that matched their given field). One issue of note is that the named entity tags from Spacy do not seem to be fully up to date with the inventory of company names, as there remained numerous instances of NE's for companies in all of the results.

I then tried increasing the number of topics further to 30 given we were dealing with a dataset of thousands of job descriptions for a wide variety of different feilds (see results below). Interestingly, when I ran it with 30 topics and 20 passes, while the results were less of a random grabbag of words that were often totally unrelated, there were none that were quite as coherent as the nursing related clusters above (there was a nursing related cluster, but not quite as much coherent of a vocab)



In [ ]:
# Question 1: LDA Topics
lda_model.print_topics(num_topics=30, num_words=10)

[(0,
  '0.015*"skf" + 0.007*"specimen" + 0.006*"fedex" + 0.005*"autism" + 0.005*"nordstrom" + 0.004*"napa" + 0.004*"ace" + 0.004*"mom" + 0.003*"aba" + 0.003*"seattle"'),
 (1,
  '0.011*"galt" + 0.010*"credit" + 0.007*"sale" + 0.007*"figure" + 0.007*"earning" + 0.007*"saas" + 0.007*"owner" + 0.006*"personal" + 0.006*"membership" + 0.005*"careerstaff"'),
 (2,
  '0.012*"trading" + 0.005*"trader" + 0.004*"ios" + 0.003*"thrivework" + 0.002*"eci" + 0.002*"pcu" + 0.002*"altame" + 0.002*"massage" + 0.002*"quantitative" + 0.002*"hydro"'),
 (3,
  '0.011*"beauty" + 0.007*"trailer" + 0.006*"ulta" + 0.006*"bakery" + 0.005*"req" + 0.005*"tradesman" + 0.003*"hair" + 0.003*"street" + 0.003*"foodservice" + 0.003*"datadog"'),
 (4,
  '0.006*"leasing" + 0.004*"associatesassist" + 0.004*"bayada" + 0.002*"delta" + 0.002*"pierce" + 0.002*"tpm" + 0.002*"gate" + 0.002*"csat" + 0.002*"westside" + 0.002*"mcc"'),
 (5,
  '0.007*"hardware" + 0.006*"coordinator" + 0.006*"chain" + 0.006*"logistic" + 0.005*"troubleshoo

### Question 2

Try to devise a specific query where TF-IDF and Doc2Vec returned very different top matches. What does this tell you about the difference between lexical similarity (shared words) and semantic similarity (shared meaning)? Include the query with your response.

## Answer 2:
The query `query_q2 = ["developer","information","application","design"]` results in differing results (see cell below). This is clearly due to the fact that Doc2Vec recognizes synonyms and related terms while TF-IDF must match exact terms (or lemmas thereof).

In the TF-IDF results, the top match was in the healthcare field and the other two were in the tech/software development field (which is what the target was actually), whereas in the Doc2Vec, the top 2 results are tech/software development though the 3rd was not.

So neither was perfect, but none of the results were the same (as they were in the original test) and the top result was clearly wrong in the TF-IDF test.

In [ ]:
query_q2 = [
    "developer",
    "information",
    "application",
    "design"
]

#tf-idf
print("TF-IDF")
query_bow_q2 = docs_dict.doc2bow(query_q2)
query_tfidf_q2 = tfidf_model[query_bow_q2]
index_tfidf_q2 = similarities.MatrixSimilarity(tfidf_corpus)
sims_tfidf_q2 = index_tfidf[query_tfidf_q2] # results in np array of scores

sorted_sims_tfidf_q2 = sorted(enumerate(sims_tfidf_q2), key=lambda sim: -sim[1])
for rank in range(3):
  doc_index_q2 = sorted_sims_tfidf_q2[rank][0]
  sim_score_q2 = sorted_sims_tfidf_q2[rank][1]
  print(f"\n\nRank {rank + 1} (Score: {sim_score:.4f}):")
  print("\n".join(textwrap.wrap(job_descriptions[doc_index_q2])))

#Doc2Vec
print("Doc2Vec")
query_vec_q2 = d2v_model.infer_vector(query_q2)
sims_d2v = d2v_model.dv.most_similar([query_vec_q2], topn=3)
for rank, (tag, sim_score) in enumerate(sims_d2v[:3]):
  doc_index = int(tag)
  print(f"\n\nRank {rank + 1} (Score: {sim_score_q2:.4f})")
  print(f"\n".join(textwrap.wrap(job_descriptions[doc_index])))

TF-IDF


Rank 1 (Score: 0.6766):
Job Descriptions: We are looking for a HealthCare Developer Manager.
You will play a pivotal role in driving the development and
implementation of healthcare web applications, ensuring high-quality
code, and guiding the project's technical direction while also
managing a team of developers.  What you can expect to focus on in
this role:Oversee developers and manage developers' work to ensure it
meets timelines for QE.Gate Keeper for Final Code reviews before
moving stories to QE.Resource should have 10+ Years of experience Gate
Keeper for deployment checklist /code mergers.To be considered for
this role, we would love you to have the following
knowledge/experience:Preferable knowledge of (Provider, Edifecs,
Facets, MDP, HIE) Proficient experience in (Guiding Care, SAP,
Portals, and Dynamics) Experience with agile development methodologies
(e.g., Scrum, Kanban) is preferred.Ability to work independently and
manage multiple priorities in a fast-paced envi

### Question 3

In working with these unsupervised methods, how did the lack of labeled ground truth affect your ability to evaluate which method was "better"? What strategies did you use to judge model quality without explicit labels?

## Answer 3:

Without a ground truth to test against, I relied upon manually checking the clusters and rankings. With regards to the Similarity of job postings to the queries, this was managable because we only chose to check the top 3 results, given there were thousands of posts in the dataset, this would not be reasonable to check all of the rankings this way.

Topic clustering is managable to check manually but there is no scientific way to rate the quality beyond just reading the clusters and deciding which seem to make sense and which don't.

### Question 4

When working with the pre-trained GloVe embeddings, where did they seem to capture intuitive relationships between terms, and where did they break down? What might this suggest about using general-purpose embeddings for specialized domains?

## Answer 4:
For Vector Arithmatic the result for the first try was reasonable (i.e. "architect" is a good possible result for `"engineer", "design" - "software"`), however for the second try (`"data", "technical" - "engineer"`), the results are somewhat nonsensical and none are actual jobs or (Ideally they should've produced a job that requires analysis of data but not an engineer). This may be due to the training data for glove being not from the domain of jobs, but from Wikipedia.

 For Odd-one out: in the example of "`words1`" (below), where all terms are clearly related to tech/software industry with the exception of "snake" that worked very well. However in the second "`words2`" where, "education" seems to be the most likely mis-match, it choses "pipeline". However, in `words2`, it is possible that all of them would belong in some contexts (i.e. software design ***for*** *education*). This shows that the use of `glove_vectors.doesnt_match()` only really works in more obvious cases.

In [ ]:
#  Vector Arithmatic 1
result = glove_vectors.most_similar(positive=["engineer", "design"], negative=["software"], topn=5)
for word, similarity_score in result:
    print(f"{similarity_score:.3f} | {word}")

0.743 | architect
0.673 | commissioned
0.630 | sculptor
0.616 | conductor
0.614 | distinguished


In [ ]:
# Odd-one Out 1
print(words1)
print(glove_vectors.doesnt_match(words1))

['python', 'software', 'engineer', 'data', 'snake']
snake


In [ ]:
#  Vector Arithmatic 2
result2 = glove_vectors.most_similar(positive=["data", "technical"], negative=["engineer"], topn=5)
for word, similarity_score in result2:
    print(f"{similarity_score:.3f} | {word}")

0.732 | specific
0.721 | scope
0.710 | analysis
0.704 | assessing
0.700 | evaluating


In [ ]:
# Odd-one Out 2
print(words2)
print(glove_vectors.doesnt_match(words2))

['pipeline', 'design', 'workflow', 'database', 'education']
pipeline


### Question 5

Imagine you had labeled data for your task (e.g. document categories or relevance scores). How would that change the role of these unsupervised techniques in your workflow? Describe how you might combine unsupervised methods (LDA, embeddings) with supervised models to build a more feature-rich or robust system.

## Answer 5:

Labeled data would enable the training, and subsequent testing of a classifier.  Without the labels, i.e. in unsupervised scenarios, you can only cluster and find patterns in documents, not predict categories for new documents the model hasn't seen.

The vector outputs of LDA, Doc2Vec, and TF-IDF and GloVe could be reused and combined into a single vector then fed into a supervised model such as a Naive Bayes or SVM as feature vector. Doing this, each of these features' shortcomings could be enhanced.

On the document level, LDA only groups vocabulary from all documents in a corpus into *unlabeled*, numbered clusters (i.e. it provides a probability distribution over unlabeled topic clusters for each given document). Since the input of LDA's probabilistic model is based on word counts and co-occurrence (i.e. TF-IDF or Bag of Words input), which do not account for synonyms or alternate phrasing, combining LDA this with Doc2Vec's dense semantic embedding for a document, which is not vocabulary dependent, combining these two features would be beneficial.

Additionally, on the word-level, along the same lines, given that TF-IDF does not capture synonymous language, the TF-IDF for a vocabulary in a given set of documents could be enhanced by combining with GloVe embeddings (as averaged vectors).

### Note to self:
* TODO: combine LDA + Doc2Vec + TF-IDF + GloVe vectors and compare clusters

---

## Notebook Submission

Before submitting:
- Ensure all code cells have been executed and display output
- Verify that all written response cells contain your analysis

See Canvas for additional submission instructions.